In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
from collections import Counter
torch.__version__

'2.9.1'

5.2 Pytorch处理结构化数据
简介
在介绍之前，我们首先要明确下什么是结构化的数据。结构化数据，可以从名称中看出，是高度组织和整齐格式化的数据。它是可以放入表格和电子表格中的数据类型。对我们来说，结构化数据可以理解为就是一张2维的表格，例如一个csv文件，就是结构化数据，在英文一般被称作Tabular Data或者叫 structured data，下面我们来看一下结构化数据的例子。

一下文件来自于fastai的自带数据集：
https://github.com/fastai/fastai/blob/master/examples/tabular.ipynb
fastai样例在这里


数据预处理
我们拿到的结构化数据，一般都是一个csv文件或者数据库中的一张表格，所以对于结构化的数据，我们直接使用pasdas库处理就可以了

In [3]:
#读入文件
df = pd.read_csv('./data/adult.csv')
#salary是这个数据集最后要分类的结果
df['salary'].unique()

array(['>=50k', '<50k'], dtype=object)

In [4]:
#查看数据类型
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,49,Private,101320,Assoc-acdm,12.0,Married-civ-spouse,NaN,Wife,White,Female,0,1902,40,United-States,>=50k
1,44,Private,236746,Masters,14.0,Divorced,Exec-managerial,Not-in-family,White,Male,10520,0,45,United-States,>=50k
2,38,Private,96185,HS-grad,NaN,Divorced,NaN,Unmarried,Black,Female,0,0,32,United-States,<50k
3,38,Self-emp-inc,112847,Prof-school,15.0,Married-civ-spouse,Prof-specialty,Husband,Asian-Pac-Islander,Male,0,0,40,United-States,>=50k
4,42,Self-emp-not-inc,82297,7th-8th,NaN,Married-civ-spouse,Other-service,Wife,Black,Female,0,0,50,United-States,<50k


In [5]:
#pandas的describe可以告诉我们整个数据集的大概结构，是非常有用的
df.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,32561.000000,3.256100e+04,32074.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.079815,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572999,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [6]:
#查看一共有多少数据
len(df)

32561

对于模型的训练，只能够处理数字类型的数据，所以这里面我们首先要将数据分成三个类别
- 训练的结果标签：即训练的结果，通过这个结果我们就能够明确的知道我们这次训练的任务是什么，是分类的任务，还是回归的任务。
- 分类数据：这类的数据是离散的，无法通过直接输入到模型中进行训练，所以我们在预处理的时候需要优先对这部分进行处理，这也是数据预处理的主要工作之一
- 数值型数据：这类数据是直接可以输入到模型中的，但是这部分数据有可能还是离散的，所以如果需要也可以对其进行处理，并且处理后会对训练的精度有很大的提升，这里暂且不讨论

In [7]:
#训练结果
result_var = 'salary'
#分类型数据
cat_names = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race','sex','native-country']
#数值型数据
cont_names = ['age', 'fnlwgt', 'education-num','capital-gain','capital-loss','hours-per-week']

人工确认完数据类型后，我们可以看一下分类类型数据的数量和分布情况

In [8]:
# print(df['workclass'])
for col in df.columns:
    if col in cat_names:
        # print(type(df[col]))  # <class 'pandas.core.series.Series'>
        # 0        4
        # 1        4
        # ...
        # print(df[col])
        # Counter 是 Python 内置库 collections 中的一个工具类，专门用于可迭代对象的元素频次统计，会返回一个字典格式的结果：{元素: 出现次数}。
        # 等价替代：df[col].value_counts()（pandas 方法），但 Counter 更轻量、支持更多灵活操作。
        print('value_counts= ',df[col].value_counts())
        # ccol=Counter(df[col])
        # print(col,len(ccol),ccol)
        print("\r\n")

value_counts=  workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64


value_counts=  education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64


value_counts=  marital-status
Married-civ-spouse       14976
Never-married            10683
Divorced                  4443
Separated                 1025
Widowed                    993
Married-spouse-absent      418
Married-AF-spouse           23
Name: count, dtype: int64


value_counts=  occupation
Prof-sp

下一步就是要将分类型数据转成数字型数据，在这部分里面，我们还做了对于缺失数据的填充

In [9]:
label_mappings = {}  # 存储{列名: {编码数字: 原始类别}}
for col in df.columns:
    if col in cat_names:
        # 填充缺失值并赋值回原列
        df[col] = df[col].fillna('---')
        le = LabelEncoder()
        # 查看类的所属模块（静态）   sklearn.preprocessing._label
        # print("LabelEncoder类的模块：", LabelEncoder.__module__)
        # 查看实例的所属模块（动态，和类一致）  sklearn.preprocessing._label
        # print("le实例的类模块：", le.__class__.__module__)
        # sklearn/preprocessing/__init__.py 中写了 from ._label import LabelEncoder，所以你从 sklearn.preprocessing 导入的 LabelEncoder，实际是 _label.py 中的类。
        # print(type(le)) # <class 'sklearn.preprocessing._label.LabelEncoder'>

        # 将分类字符串（如Private/---）映射为 0、1、2... 等整数（每个唯一值对应一个数字）；
        df[col] = le.fit_transform(df[col].astype(str))
        # 保存编码映射：数字→原始类别
        # le.classes_ = [' ?' ' Federal-gov' ' Local-gov' ' Never-worked' ' Private' ' Self-emp-inc' ' Self-emp-not-inc' ' State-gov' ' Without-pay']
        print("le.classes_ =", le.classes_)
        encoded_nums = le.transform(le.classes_)
        # le.transform(le.classes_) = [0 1 2 3 4 5 6 7 8]
        print("le.transform(le.classes_) =", encoded_nums)
        # zip(编码数字, 原始类别) → 配对 “数字 - 类别”
        # zip(a, b) 会把两个列表的元素按位置一一配对，生成可迭代的元组
        # pairs = zip(encoded_nums, le.classes_)
        # zip后的配对结果 = [(np.int64(0), ' Divorced'), (np.int64(1), ' Married-AF-spouse')...]
        # print("zip后的配对结果 =", list(pairs))
        # dict() 把 zip 生成的元组对转为字典，键是编码数字，值是原始类别
        label_mappings[col] = dict(zip(le.transform(le.classes_), le.classes_))
        print('label_mappings[col]=',label_mappings[col])
        # print(df[col])
    if col in cont_names:
        df[col]=df[col].fillna(0)

le.classes_ = [' ?' ' Federal-gov' ' Local-gov' ' Never-worked' ' Private'
 ' Self-emp-inc' ' Self-emp-not-inc' ' State-gov' ' Without-pay']
le.transform(le.classes_) = [0 1 2 3 4 5 6 7 8]
label_mappings[col]= {np.int64(0): ' ?', np.int64(1): ' Federal-gov', np.int64(2): ' Local-gov', np.int64(3): ' Never-worked', np.int64(4): ' Private', np.int64(5): ' Self-emp-inc', np.int64(6): ' Self-emp-not-inc', np.int64(7): ' State-gov', np.int64(8): ' Without-pay'}
le.classes_ = [' 10th' ' 11th' ' 12th' ' 1st-4th' ' 5th-6th' ' 7th-8th' ' 9th'
 ' Assoc-acdm' ' Assoc-voc' ' Bachelors' ' Doctorate' ' HS-grad'
 ' Masters' ' Preschool' ' Prof-school' ' Some-college']
le.transform(le.classes_) = [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
label_mappings[col]= {np.int64(0): ' 10th', np.int64(1): ' 11th', np.int64(2): ' 12th', np.int64(3): ' 1st-4th', np.int64(4): ' 5th-6th', np.int64(5): ' 7th-8th', np.int64(6): ' 9th', np.int64(7): ' Assoc-acdm', np.int64(8): ' Assoc-voc', np.int64(9): ' Bachel

上面的代码中：

我们首先使用了pandas的fillna函数对分类的数据做了空值的填充，这里面标识成一个与其他现有值不一样的值就可以，这里面我使用的三个中划线 --- 作为标记，然后就是使用了sklearn的LabelEncoder函数进行了数据的处理

然后有对我们的数值型的数据做了0填充的处理，对于数值型数据的填充，也可以使用平均值，或者其他方式填充，这个不是我们的重点，就不详细说明了。

In [10]:
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,49,4,101320,7,12.0,2,15,5,4,0,0,1902,40,39,>=50k
1,44,4,236746,12,14.0,0,4,1,4,1,10520,0,45,39,>=50k
2,38,4,96185,11,0.0,0,15,4,2,0,0,0,32,39,<50k
3,38,5,112847,14,15.0,2,10,0,1,1,0,0,40,39,>=50k
4,42,6,82297,5,0.0,2,8,5,2,0,0,0,50,39,<50k


数据处理完成后可以看到，现在所有的数据都是数字类型的了，可以直接输入到模型进行训练了.

In [11]:
#分割下训练数据和标签
Y = df['salary']
Y_label = LabelEncoder()
# 拟合 + 编码 变为0或1 。 拟合从0开始，到类别-1,
Y=Y_label.fit_transform(Y)
Y

array([1, 1, 0, ..., 1, 0, 0], shape=(32561,))

In [12]:
X=df.drop(columns=result_var)
X

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,49,4,101320,7,12.0,2,15,5,4,0,0,1902,40,39
1,44,4,236746,12,14.0,0,4,1,4,1,10520,0,45,39
2,38,4,96185,11,0.0,0,15,4,2,0,0,0,32,39
3,38,5,112847,14,15.0,2,10,0,1,1,0,0,40,39
4,42,6,82297,5,0.0,2,8,5,2,0,0,0,50,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,36,4,297449,9,13.0,0,10,1,4,1,14084,0,40,39
32557,23,0,123983,9,13.0,4,0,3,3,1,0,0,40,39
32558,53,4,157069,7,12.0,2,7,0,4,1,0,0,40,39
32559,32,2,217296,11,9.0,2,14,5,4,0,4064,0,22,39


以上，基本的数据预处理已经完成了，上面展示的只是一些必要的处理，如果要提高训练准确率还有很多技巧，这里就不详细说明了。
定义数据集
要使用pytorch处理数据，肯定要使用Dataset进行数据集的定义，下面定义一个简单的数据集

In [13]:
class tabularDataset(Dataset):
    def __init__(self, X, Y):
        # X 是 pandas DataFrame/Series，需要 .values 转成 NumPy 数组；而 Y 已经是 NumPy 数组（或一维可索引对象），无需再转。
        self.x = X.values
        self.y = Y
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return (self.x[idx], self.y[idx])

In [14]:
train_ds = tabularDataset(X, Y)

可以直接使用索引访问定义好的数据集中的数据

In [15]:
train_ds[0]

(array([4.9000e+01, 4.0000e+00, 1.0132e+05, 7.0000e+00, 1.2000e+01,
        2.0000e+00, 1.5000e+01, 5.0000e+00, 4.0000e+00, 0.0000e+00,
        0.0000e+00, 1.9020e+03, 4.0000e+01, 3.9000e+01]),
 np.int64(1))

定义模型
数据已经准备完毕了，下一步就是要定义我们的模型了，这里使用了3层线性层的简单模型作为处理

In [16]:
class tabularModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1 = nn.Linear(14, 500)
        self.lin2 = nn.Linear(500, 100)
        # 单标签分类,每个样本输出 2 个类别的「原始得分」）,这些得分不是概率（未做 softmax），但得分越高，模型认为该样本属于这个类别的概率越大；我们需要从这 2 个得分中，选出「得分最高的那个类别索引」—— 这就是 torch.max 的作用。
        self.lin3 = nn.Linear(100, 2)
        #nn.BatchNorm1d(14)	输入层批量归一化：标准化输入特征，加速收敛，减少初始化敏感；
        #nn.BatchNorm1d(500/100)	隐藏层批量归一化：缓解梯度消失，允许使用更大学习率；
        self.bn_in = nn.BatchNorm1d(14)
        self.bn1 = nn.BatchNorm1d(500)
        self.bn2 = nn.BatchNorm1d(100)

    def forward(self,x_in):
        #print(x_in.shape)
        x = self.bn_in(x_in)
        # ReLU 是 Rectified Linear Unit（修正线性单元）的缩写，数学公式极其简单：ReLU(x)=max(0,x)
        # 当输入 x > 0 时，输出等于输入（y = x）；
        # 当输入 x ≤ 0 时，输出为 0（相当于 “关闭” 这个神经元）。
        # 变体              公式               特点                        代码使用
        # ReLU（基础）      max(0,x)           简单高效，有死亡 ReLU 风险     F.relu(x)
        # Leaky ReLU       max(αx,x)（α=0.01）负数输入保留小梯度，避免死亡     ReLUF.leaky_relu(x, negative_slope=0.01)
        # ReLU6            min(max(0,x),6)    限制最大输出为 6，适合量化模型  F.relu6(x)GELUx⋅Φ(x)（Φ 是高斯分布 CDF）更平滑，BERT 等大模型常用     F.gelu(x)
        x = F.relu(self.lin1(x))
        x = self.bn1(x)
        #print(x)
        
        
        x = F.relu(self.lin2(x))
        x = self.bn2(x)
        #print(x)
        
        x = self.lin3(x)
        #Sigmoid（也叫 Logistic 函数）的数学公式：
        # # 简洁版（推荐，代码注释用）
        # σ(x) = 1 / (1 + e^(-x))
        # 或
        # Sigmoid(x) = 1 / (1 + exp(-x))
        # 当输入 x → +∞（极大正数）时，输出趋近于 1；
        # 当输入 x = 0 时，输出等于 0.5；
        # 当输入 x → -∞（极小负数）时，输出趋近于 0；
        # 函数曲线是 “S 型”，平滑且单调递增。
        x=torch.sigmoid(x)
        return x

在定义模型的时候看到了我们加入了Batch Normalization来做批量的归一化：
批量归一化的内容请见这篇文章：https://mp.weixin.qq.com/s/FFLQBocTZGqnyN79JbSYcQ

或者扫描这个二维码，在微信中查看:
![](https://raw.githubusercontent.com/zergtant/pytorch-handbook/master/deephub.jpg)

训练

In [17]:
#训练前指定使用的设备
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device('mps')  # Apple Silicon (M1/M2/M3)
else:
    DEVICE = torch.device('cpu')

In [18]:
#损失函数
criterion =nn.CrossEntropyLoss()

In [19]:
#实例化模型
model = tabularModel().to(DEVICE)
print(model)

tabularModel(
  (lin1): Linear(in_features=14, out_features=500, bias=True)
  (lin2): Linear(in_features=500, out_features=100, bias=True)
  (lin3): Linear(in_features=100, out_features=2, bias=True)
  (bn_in): BatchNorm1d(14, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn1): BatchNorm1d(500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)


In [20]:
#测试模型是否没问题
rn=torch.rand(3,14).to(DEVICE)
model(rn)

tensor([[0.4213, 0.6817],
        [0.4905, 0.3254],
        [0.5591, 0.4510]], device='mps:0', grad_fn=<SigmoidBackward0>)

In [21]:
#学习率
LEARNING_RATE=0.01
#BS
batch_size = 1024
#优化器
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [22]:
#DataLoader加载数据
train_dl = DataLoader(train_ds, batch_size=batch_size,shuffle=True)

以上的基本步骤是每个训练过程都需要的，所以就不多介绍了，下面开始进行模型的训练

In [25]:
%%time
model.train()
#训练10轮
TOTAL_EPOCHS=100

for epoch in range(TOTAL_EPOCHS):
    #记录损失函数
    losses = [];
    for i, (x, y) in enumerate(train_dl):
        x = x.float().to(DEVICE) #输入必须未float类型
        y = y.long().to(DEVICE) #结果标签必须未long类型
        #清零
        optimizer.zero_grad()
        outputs = model(x)
        #计算损失函数
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.cpu().data.item())
    print ('Epoch : %d/%d,   Loss: %.4f'%(epoch+1, TOTAL_EPOCHS, np.mean(losses)))

Epoch : 1/100,   Loss: 0.4099
Epoch : 2/100,   Loss: 0.4105
Epoch : 3/100,   Loss: 0.4105
Epoch : 4/100,   Loss: 0.4096
Epoch : 5/100,   Loss: 0.4105
Epoch : 6/100,   Loss: 0.4085
Epoch : 7/100,   Loss: 0.4103
Epoch : 8/100,   Loss: 0.4081
Epoch : 9/100,   Loss: 0.4078
Epoch : 10/100,   Loss: 0.4096
Epoch : 11/100,   Loss: 0.4079
Epoch : 12/100,   Loss: 0.4070
Epoch : 13/100,   Loss: 0.4076
Epoch : 14/100,   Loss: 0.4075
Epoch : 15/100,   Loss: 0.4054
Epoch : 16/100,   Loss: 0.4081
Epoch : 17/100,   Loss: 0.4085
Epoch : 18/100,   Loss: 0.4069
Epoch : 19/100,   Loss: 0.4084
Epoch : 20/100,   Loss: 0.4075
Epoch : 21/100,   Loss: 0.4085
Epoch : 22/100,   Loss: 0.4066
Epoch : 23/100,   Loss: 0.4059
Epoch : 24/100,   Loss: 0.4059
Epoch : 25/100,   Loss: 0.4051
Epoch : 26/100,   Loss: 0.4053
Epoch : 27/100,   Loss: 0.4059
Epoch : 28/100,   Loss: 0.4060
Epoch : 29/100,   Loss: 0.4063
Epoch : 30/100,   Loss: 0.4051
Epoch : 31/100,   Loss: 0.4030
Epoch : 32/100,   Loss: 0.4038
Epoch : 33/100,  

训练完成后我们可以看一下模型的准确率

In [28]:
with torch.no_grad():
    model.eval()
    correct = 0
    total = 0
    for i,(x, y) in enumerate(train_dl):
        x = x.float().to(DEVICE)
        y = y.long().to(DEVICE)
        outputs = model(x)
        # 取预测类别（dim=1表示按类别维度取最大值）
        # torch.max 返回一个元组 (max_values, max_indices)：
        # max_values：沿 dim=1 取到的「最大值」（即每个样本的最高类别得分）；
        # max_indices：沿 dim=1 取到的「最大值对应的索引」（即每个样本的预测类别）。
        _, predicted = torch.max(outputs, dim=1)
         # 累加总样本数（y.size(0)是批次样本数）
        total += y.size(0)
        # predicted 是 torch.LongTensor 类型（整数），和分类任务的标签类型（long）完全匹配，因此可以直接和标签 y 比较（predicted == y）
        # 累加正确数：转成标量避免张量累加错误
        correct += (predicted == y).sum().item()
print('准确率: %.4f %%' % (100 * correct / total))
# 计算准确率（转浮点保证除法精度）
accuracy = 100.0 * correct / total
# 更清晰的打印格式（保留4位小数）
print(f'准确率: {accuracy:.4f} %')

准确率: 92.0580 %
准确率: 92.0580 %


以上就是基本的流程了